# Aula 19 - Notebook: Árvore Geradora Mínima (Kruskal e Prim)

**Projeto:** SCADA-Core Automática — GRUPO 4  
**Unidade:** Planta Automatizada de Seleção e Classificação Óptica de Grãos com Separação Pneumática Tripla  
**Engenharia de Automação e Controle** — ECAA08  

---

### 1. Objetivos do Notebook
Neste notebook implementamos e validamos os algoritmos de **Kruskal** (com *Union-Find*) e **Prim** (com *Min-Heap*) aplicados ao projeto da **rede de cabeamento industrial SCADA** da planta:
1. Modelagem dos $11$ nós de automação e $19$ trechos candidatos de eletrocalha;
2. Execução passo a passo do algoritmo de **Kruskal** com detecção de ciclos via *Disjoint Set Union* (DSU);
3. Execução do algoritmo de **Prim** partindo do painel central `CLP-01_Painel`;
4. Verificação formal das propriedades da MST ($|E_T| = |V| - 1$, aciclicidade e equivalência de custo mínimo);
5. Análise de contingência $N-1$ e cálculo da **Segunda Melhor Árvore Geradora** para planejamento de redundância.

### 2. Funções Auxiliares de Formatação Tabular ASCII

In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

### 3. Modelagem da Rede Candidata de Cabeamento da Planta (Grupo 4)

In [2]:
from typing import List, Dict, Tuple, Any
import heapq

NOS_PLANTA = [
    "CLP-01_Painel",
    "SEG-901_Emergencia",
    "FN-101_Funil",
    "ALIM-101_Calha",
    "EST-201_Esteira",
    "BAL-301_Balanca",
    "VIS-401_Visao",
    "EJE-602_EstacaoB",
    "EJE-603_EstacaoC",
    "COMP-601_Pneumatica",
    "SILOS-700_Coleta",
]

ARESTAS_CANDIDATAS = [
    ("CLP-01_Painel", "SEG-901_Emergencia", 3.5, "Eletrocalha Painel -> Emergencia (XA-901)"),
    ("CLP-01_Painel", "FN-101_Funil", 8.2, "Eletrocalha Principal -> Funil (LIT-101)"),
    ("CLP-01_Painel", "EST-201_Esteira", 6.4, "Eletrocalha Central -> Motor (ST-201/JI-201)"),
    ("CLP-01_Painel", "VIS-401_Visao", 9.8, "Eletrocalha Optica Direta (XS-401/KSA-401)"),
    ("CLP-01_Painel", "COMP-601_Pneumatica", 7.1, "Eletrocalha -> Central Pneumatica (PT-601)"),
    ("SEG-901_Emergencia", "FN-101_Funil", 5.6, "Derivacao Seguranca -> Funil"),
    ("SEG-901_Emergencia", "ALIM-101_Calha", 6.9, "Derivacao Seguranca -> Alimentador"),
    ("FN-101_Funil", "ALIM-101_Calha", 2.4, "Interligacao Funil -> Alimentador (c_ALIM)"),
    ("ALIM-101_Calha", "EST-201_Esteira", 3.1, "Interligacao Alimentador -> Esteira (c_EST)"),
    ("EST-201_Esteira", "BAL-301_Balanca", 3.8, "Leito Esteira -> Balanca (WT-301)"),
    ("BAL-301_Balanca", "VIS-401_Visao", 4.2, "Leito Balanca -> Tunel Visao"),
    ("VIS-401_Visao", "EJE-602_EstacaoB", 3.9, "Barramento Visao -> Ejetor B (FY-602)"),
    ("EJE-602_EstacaoB", "EJE-603_EstacaoC", 2.1, "Barramento Ejetor B -> Ejetor C (FY-603)"),
    ("EJE-602_EstacaoB", "COMP-601_Pneumatica", 4.7, "Ramal Ar/Sinal -> Ejetor B"),
    ("EJE-603_EstacaoC", "COMP-601_Pneumatica", 4.3, "Ramal Ar/Sinal -> Ejetor C"),
    ("EJE-603_EstacaoC", "SILOS-700_Coleta", 4.9, "Cabeceira -> Concentrador Silos (LIT-701..703)"),
    ("EJE-602_EstacaoB", "SILOS-700_Coleta", 6.2, "Derivacao Ejetor B -> Silos"),
    ("COMP-601_Pneumatica", "SILOS-700_Coleta", 5.8, "Derivacao Compressor -> Silos"),
    ("BAL-301_Balanca", "COMP-601_Pneumatica", 7.5, "Cruzamento Inferior Balanca -> Compressor"),
]

custo_malha_completa = round(sum(a[2] for a in ARESTAS_CANDIDATAS), 2)
print(f"Total de nos (|V|)         : {len(NOS_PLANTA)}")
print(f"Trechos candidatos (|E|)   : {len(ARESTAS_CANDIDATAS)}")
print(f"Comprimento malha completa : {custo_malha_completa:.2f} m")

Total de nos (|V|)         : 11
Trechos candidatos (|E|)   : 19
Comprimento malha completa : 100.40 m


### 4. Algoritmo de Kruskal com Union-Find (DSU)

In [3]:
class UnionFind:
    """Estrutura Disjoint Set Union com path compression e union by rank."""
    def __init__(self, elementos: List[str]):
        self.pai = {e: e for e in elementos}
        self.rank = {e: 0 for e in elementos}

    def find(self, x: str) -> str:
        if self.pai[x] != x:
            self.pai[x] = self.find(self.pai[x])
        return self.pai[x]

    def union(self, x: str, y: str) -> bool:
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return False
        if self.rank[rx] < self.rank[ry]:
            self.pai[rx] = ry
        elif self.rank[rx] > self.rank[ry]:
            self.pai[ry] = rx
        else:
            self.pai[ry] = rx
            self.rank[rx] += 1
        return True

def kruskal_mst(nos: List[str], arestas: List[Tuple[str, str, float, str]]):
    uf = UnionFind(nos)
    mst = []
    historico = []
    custo_total = 0.0

    for idx, (u, v, w, desc) in enumerate(sorted(arestas, key=lambda x: x[2]), start=1):
        aceita = uf.union(u, v)
        if aceita:
            mst.append((u, v, w, desc))
            custo_total += w
        historico.append({
            "Passo": idx,
            "Origem": u,
            "Destino": v,
            "Peso (m)": f"{w:.1f}",
            "Decisao": "ACEITA" if aceita else "REJEITADA (ciclo)",
            "Acumulado (m)": f"{custo_total:.1f}",
        })
        if len(mst) == len(nos) - 1:
            break

    return mst, round(custo_total, 4), historico

mst_kruskal, custo_kruskal, hist_kruskal = kruskal_mst(NOS_PLANTA, ARESTAS_CANDIDATAS)
print("=== EXECUCAO PASSO A PASSO: KRUSKAL ===")
print(formatar_tabela(hist_kruskal))
print(f"\nCusto minimo (Kruskal): {custo_kruskal:.2f} m | Arestas selecionadas: {len(mst_kruskal)}")

=== EXECUCAO PASSO A PASSO: KRUSKAL ===
Passo | Origem             | Destino             | Peso (m) | Decisao           | Acumulado (m)
------+--------------------+---------------------+----------+-------------------+--------------
1     | EJE-602_EstacaoB   | EJE-603_EstacaoC    | 2.1      | ACEITA            | 2.1          
2     | FN-101_Funil       | ALIM-101_Calha      | 2.4      | ACEITA            | 4.5          
3     | ALIM-101_Calha     | EST-201_Esteira     | 3.1      | ACEITA            | 7.6          
4     | CLP-01_Painel      | SEG-901_Emergencia  | 3.5      | ACEITA            | 11.1         
5     | EST-201_Esteira    | BAL-301_Balanca     | 3.8      | ACEITA            | 14.9         
6     | VIS-401_Visao      | EJE-602_EstacaoB    | 3.9      | ACEITA            | 18.8         
7     | BAL-301_Balanca    | VIS-401_Visao       | 4.2      | ACEITA            | 23.0         
8     | EJE-603_EstacaoC   | COMP-601_Pneumatica | 4.3      | ACEITA            | 27.3         


### 5. Algoritmo de Prim (A partir do Painel Central `CLP-01_Painel`)

In [4]:
def prim_mst(nos: List[str], arestas: List[Tuple[str, str, float, str]], raiz: str):
    adj: Dict[str, List[Tuple[float, str, str]]] = {n: [] for n in nos}
    for u, v, w, desc in arestas:
        adj[u].append((w, v, desc))
        adj[v].append((w, u, desc))

    visitados = {raiz}
    heap = []
    for w, v, desc in adj[raiz]:
        heapq.heappush(heap, (w, raiz, v, desc))

    mst = []
    historico = []
    custo_total = 0.0
    passo = 1

    while heap and len(visitados) < len(nos):
        w, u, v, desc = heapq.heappop(heap)
        if v in visitados:
            continue
        visitados.add(v)
        mst.append((u, v, w, desc))
        custo_total += w
        historico.append({
            "Ordem": passo,
            "De (na MST)": u,
            "Para (Novo No)": v,
            "Peso (m)": f"{w:.1f}",
            "Acumulado (m)": f"{custo_total:.1f}",
        })
        passo += 1
        for nw, nv, ndesc in adj[v]:
            if nv not in visitados:
                heapq.heappush(heap, (nw, v, nv, ndesc))

    return mst, round(custo_total, 4), historico

mst_prim, custo_prim, hist_prim = prim_mst(NOS_PLANTA, ARESTAS_CANDIDATAS, "CLP-01_Painel")
print("=== EXECUCAO PASSO A PASSO: PRIM (Raiz: CLP-01_Painel) ===")
print(formatar_tabela(hist_prim))
print(f"\nCusto minimo (Prim): {custo_prim:.2f} m | Arestas selecionadas: {len(mst_prim)}")

=== EXECUCAO PASSO A PASSO: PRIM (Raiz: CLP-01_Painel) ===
Ordem | De (na MST)        | Para (Novo No)      | Peso (m) | Acumulado (m)
------+--------------------+---------------------+----------+--------------
1     | CLP-01_Painel      | SEG-901_Emergencia  | 3.5      | 3.5          
2     | SEG-901_Emergencia | FN-101_Funil        | 5.6      | 9.1          
3     | FN-101_Funil       | ALIM-101_Calha      | 2.4      | 11.5         
4     | ALIM-101_Calha     | EST-201_Esteira     | 3.1      | 14.6         
5     | EST-201_Esteira    | BAL-301_Balanca     | 3.8      | 18.4         
6     | BAL-301_Balanca    | VIS-401_Visao       | 4.2      | 22.6         
7     | VIS-401_Visao      | EJE-602_EstacaoB    | 3.9      | 26.5         
8     | EJE-602_EstacaoB   | EJE-603_EstacaoC    | 2.1      | 28.6         
9     | EJE-603_EstacaoC   | COMP-601_Pneumatica | 4.3      | 32.9         
10    | EJE-603_EstacaoC   | SILOS-700_Coleta    | 4.9      | 37.8         

Custo minimo (Prim): 37.80 m

### 6. Validação Formal de Equivalência e Economia de Infraestrutura

In [5]:
conj_k = {frozenset((u, v)) for u, v, _, _ in mst_kruskal}
conj_p = {frozenset((u, v)) for u, v, _, _ in mst_prim}

assert len(mst_kruskal) == len(NOS_PLANTA) - 1, "MST deve ter |V|-1 arestas"
assert len(mst_prim) == len(NOS_PLANTA) - 1, "MST deve ter |V|-1 arestas"
assert custo_kruskal == custo_prim, "Custos de Kruskal e Prim devem coincidir"
assert conj_k == conj_p, "Com pesos distintos, o conjunto de arestas da MST deve ser unico"

economia_m = round(custo_malha_completa - custo_kruskal, 2)
economia_pct = (economia_m / custo_malha_completa) * 100.0

print("VALIDACAO FORMAL APROVADA:")
print(f"  - Numero de arestas |E_T| = {len(mst_kruskal)} == |V|-1 ({len(NOS_PLANTA)-1})")
print(f"  - Kruskal == Prim?        : {conj_k == conj_p} (custo = {custo_kruskal:.2f} m)")
print(f"  - Economia de cabeamento  : {economia_m:.2f} m ({economia_pct:.2f}% de reducao)")

VALIDACAO FORMAL APROVADA:
  - Numero de arestas |E_T| = 10 == |V|-1 (10)
  - Kruskal == Prim?        : True (custo = 37.80 m)
  - Economia de cabeamento  : 62.60 m (62.35% de reducao)


### 7. Análise de Contingência ($N-1$) e Segunda Melhor Árvore Geradora
Simulamos a falha individual de cada uma das $10$ arestas da MST para identificar:
* Qual trecho alternativo assume a conexão caso o cabo se rompa;
* Qual o impacto no comprimento total e qual é a **Segunda Melhor Árvore Geradora** para planejamento de redundância no SCADA.

In [6]:
relatorio_n1 = []
segunda_melhor_custo = float("inf")
segunda_melhor_detalhe = None

for u_rem, v_rem, w_rem, desc_rem in mst_kruskal:
    sub_arestas = [a for a in ARESTAS_CANDIDATAS if not (a[0] == u_rem and a[1] == v_rem)]
    nova_mst, novo_custo, _ = kruskal_mst(NOS_PLANTA, sub_arestas)
    conectada = (len(nova_mst) == len(NOS_PLANTA) - 1)
    if conectada:
        novas = {frozenset((a[0], a[1])): a for a in nova_mst}
        antigas = {frozenset((a[0], a[1])) for a in mst_kruskal if not (a[0] == u_rem and a[1] == v_rem)}
        substituta = [v for k, v in novas.items() if k not in antigas][0]
        delta = round(novo_custo - custo_kruskal, 2)
        if novo_custo < segunda_melhor_custo:
            segunda_melhor_custo = novo_custo
            segunda_melhor_detalhe = (u_rem, v_rem, substituta[0], substituta[1], delta)
        relatorio_n1.append({
            "Aresta Rompida": f"{u_rem} -- {v_rem}",
            "Peso (m)": f"{w_rem:.1f}",
            "Substituta": f"{substituta[0]} -- {substituta[1]}",
            "Peso Sub (m)": f"{substituta[2]:.1f}",
            "Novo Custo (m)": f"{novo_custo:.1f}",
            "Delta (m)": f"+{delta:.1f}",
        })
    else:
        relatorio_n1.append({
            "Aresta Rompida": f"{u_rem} -- {v_rem}",
            "Peso (m)": f"{w_rem:.1f}",
            "Substituta": "NENHUMA (PONTO UNICO)",
            "Peso Sub (m)": "-",
            "Novo Custo (m)": "DESCONEXO",
            "Delta (m)": "-",
        })

print("=== ANALISE DE CONTINGENCIA N-1 (ROMPIMENTO DE CABOS DA MST) ===")
print(formatar_tabela(relatorio_n1))
print(f"\nSegunda Melhor MST: {segunda_melhor_custo:.2f} m "
      f"(troca {segunda_melhor_detalhe[0]}--{segunda_melhor_detalhe[1]} por "
      f"{segunda_melhor_detalhe[2]}--{segunda_melhor_detalhe[3]}, +{segunda_melhor_detalhe[4]:.1f} m)")

=== ANALISE DE CONTINGENCIA N-1 (ROMPIMENTO DE CABOS DA MST) ===
Aresta Rompida                          | Peso (m) | Substituta                              | Peso Sub (m) | Novo Custo (m) | Delta (m)
----------------------------------------+----------+-----------------------------------------+--------------+----------------+----------
EJE-602_EstacaoB -- EJE-603_EstacaoC    | 2.1      | EJE-602_EstacaoB -- COMP-601_Pneumatica | 4.7          | 40.4           | +2.6     
FN-101_Funil -- ALIM-101_Calha          | 2.4      | CLP-01_Painel -- EST-201_Esteira        | 6.4          | 41.8           | +4.0     
ALIM-101_Calha -- EST-201_Esteira       | 3.1      | CLP-01_Painel -- EST-201_Esteira        | 6.4          | 41.1           | +3.3     
CLP-01_Painel -- SEG-901_Emergencia     | 3.5      | CLP-01_Painel -- EST-201_Esteira        | 6.4          | 40.7           | +2.9     
EST-201_Esteira -- BAL-301_Balanca      | 3.8      | CLP-01_Painel -- COMP-601_Pneumatica    | 7.1          | 41.